# OmniVoice — Yoruba finetune + data-efficiency ablation (Paper-1)

Fine-tunes **k2-fsa/OmniVoice** on clean Yoruba (BibleTTS + SLR86) via the official two-stage
`examples/run_finetune.sh` pipeline, on **1×A100 80GB**.

**One knob, two jobs.** `HOURS_BUDGET = None` runs a single full finetune (pipeline validation).
Set it to `1.0 / 5.0 / 15.0 / 30.0` and re-run end-to-end to produce each point of the
**data-efficiency curve** (Paper-1 ablation): `hours → {tone_i2, cer}` on the SAME nb01 gate.

NON-autoregressive masked diffusion + rule-based duration ⇒ the Qwen3-TTS high-LR *collapse* failure
mode is structurally impossible, so we use OmniVoice's default lr=1e-5 (vs the 2e-6/2e-5 Qwen3-TTS
regime). This notebook answers: **how many clean Yoruba hours does OmniVoice need to install real tone?**

⚠️ **Attention: sdpa only.** The stock finetune config omits `attn_implementation` → defaults to
`flex_attention` (GPU-compat issues + not our allowed backend). We write `"sdpa"` explicitly.
⚠️ **License.** Stage 0 runs the **Higgs/Boson codec** (`eustlb/higgs-audio-v2-tokenizer`); its
non-commercial output-restriction clause is a flywheel-contamination risk — see `README.md` and
`BOSON_LICENSE_WAIVER_EMAIL.md`. (Finetuning OmniVoice is permitted; using its *output* to train the
duplex model is the blocked part.)

GPU: **A100 80GB (training)** — NOT a T4/L4 notebook.

## 1. Install + restart (numpy<2 pin)

In [ ]:
%pip install --quiet "numpy<2"
# OmniVoice (pulls torch/torchaudio>=2.4 + transformers>=5.3). If CUDA import breaks, pin the wheel:
# %pip install --quiet torch==2.8.0+cu128 torchaudio==2.8.0+cu128 --extra-index-url https://download.pytorch.org/whl/cu128
%pip install --quiet omnivoice
# Training + Stage-0 deps. accelerate REQUIRED (Stage 1 = accelerate launch). webdataset for shards.
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld accelerate safetensors webdataset "huggingface_hub>=0.24.0" tqdm
# Remove the Xet download backend: on a FRESH runtime hf-xet stalls at "Fetching N files: 0%" /
# "(incomplete total...) 0.00B" (the nb01 §4 hang). Here it would bite INSIDE the training
# subprocesses: Stage 0 downloads eustlb/higgs-audio-v2-tokenizer, Stage 1 downloads
# init_from_checkpoint=k2-fsa/OmniVoice — a mid-training hang wastes A100 time. Removing the package
# makes the WHOLE environment (notebook AND every subprocess) fall back to the plain HTTP backend.
%pip uninstall -y hf-xet
import os
print("Installs done. RESTARTING kernel so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)   # if Colab does not auto-restart: Runtime > Restart session

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
print("numpy", np.__version__, "OK — continue.")

## 2. Clone OmniVoice (training scripts) + `mosesdaudu001/tone-on-a-budget` (approachB gate)

In [ ]:
import os, subprocess, sys, shutil, json

OV_DIR = "/content/OmniVoice" if os.path.isdir("/content") else "/tmp/OmniVoice"
if os.path.exists(OV_DIR): shutil.rmtree(OV_DIR)
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/k2-fsa/OmniVoice.git", OV_DIR], check=True)
OV_EX = os.path.join(OV_DIR, "examples")

# --- CONFIRM-AT-RUNTIME #1: finetune entrypoints + configs exist where the recon says ---
print("examples/      :", sorted(os.listdir(OV_EX)))
print("examples/config:", sorted(os.listdir(os.path.join(OV_EX, "config"))))
#   expect run_finetune.sh ; config/{train_config_finetune.json, train_config_finetune_sdpa.json,
#   data_config_finetune.json}
assert os.path.exists(os.path.join(OV_EX, "run_finetune.sh")), "MISSING examples/run_finetune.sh — recon stale, STOP"

# --- CONFIRM-AT-RUNTIME #2: training modules importable (Stage 0 + Stage 1) ---
# The pip `omnivoice` wheel may be inference-only; PYTHONPATH=OV_DIR makes the cloned training code win.
sys.path.insert(0, OV_DIR)
for mod in ["omnivoice.scripts.extract_audio_tokens", "omnivoice.cli.train"]:
    r = subprocess.run([sys.executable, "-c", f"import importlib;importlib.import_module('{mod}')"],
                       capture_output=True, text=True, cwd=OV_DIR,
                       env=dict(os.environ, PYTHONPATH=OV_DIR))
    print(f"import {mod}:", "OK" if r.returncode == 0 else "FAIL\n" + r.stderr[-800:])
    if r.returncode != 0:
        print(f"  -> if missing, the pip wheel is inference-only; fix: %pip install -e {OV_DIR}")
    assert r.returncode == 0, f"{mod} not importable — STOP (try: pip install -e {OV_DIR})"

# --- CONFIRM-AT-RUNTIME #3: stock finetune config keys (so our derived config matches reality) ---
STOCK_CFG = os.path.join(OV_EX, "config", "train_config_finetune.json")
stock = json.load(open(STOCK_CFG))
print("\nstock train_config_finetune.json keys:", sorted(stock.keys()))
print("  init_from_checkpoint =", stock.get("init_from_checkpoint"),
      "| lr =", stock.get("learning_rate"), "| steps =", stock.get("steps"),
      "| batch_tokens =", stock.get("batch_tokens"),
      "| num_audio_codebook =", stock.get("num_audio_codebook"))
print("  attn_implementation in stock config? ->", "attn_implementation" in stock,
      "(expect False -> default flex_attention -> we override to sdpa)")
assert stock.get("init_from_checkpoint") == "k2-fsa/OmniVoice", "init_from_checkpoint drifted — re-check recon"
assert stock.get("num_audio_codebook") == 8, "codebook count drifted from 8 — re-check Higgs codec"

# --- CONFIRM-AT-RUNTIME #4: Yoruba language_id is 'yo' (same source nb01 used) ---
tsv = os.path.join(OV_DIR, "docs", "lang_id_name_map.tsv")
yo_ok = False
if os.path.exists(tsv):
    for ln in open(tsv):
        c = ln.rstrip("\n").split("\t")
        if (c and c[0] == "yo") or (len(c) > 1 and c[1].lower() == "yoruba"):
            print("Yoruba lang line:", ln.rstrip()); yo_ok = (c[0] == "yo")
assert yo_ok, "Yoruba code is not 'yo' in docs/lang_id_name_map.tsv — fix LANG_CODE before Stage 0"

# --- CONFIRM-AT-RUNTIME #5: extract CLI flags + JsonlDatasetReader audio field name ---
print("\n--- extract_audio_tokens CLI (grep add_argument) ---")
print(subprocess.run(["grep", "-n", "add_argument",
    os.path.join(OV_DIR, "omnivoice", "scripts", "extract_audio_tokens.py")],
    capture_output=True, text=True).stdout)
print("--- JsonlDatasetReader audio field (grep audio_path) ---")
print(subprocess.run(["grep", "-rn", "audio_path",
    os.path.join(OV_DIR, "omnivoice", "data", "dataset.py")],
    capture_output=True, text=True).stdout)
#   expect: meta.get("audio_path")  -> confirms our manifest key is 'audio_path'

In [ ]:
# tone-metric package (public) — replaces the old git clone + sys.path of tone_metric/
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
import os, subprocess, sys, shutil, importlib
importlib.invalidate_caches()
import tone_metric
from tone_metric import tone_probe as tp
from tone_metric import tone_f0_abs as f0a
from tone_metric.grpo_reward import RewardModels
CODE_DIR = os.path.dirname(tone_metric.__file__)   # minimal_pairs_draft.json ships as package data
print("tone_metric", tone_metric.__version__, "from", CODE_DIR)


## 3. Secrets + S3 + constants + **HOURS_BUDGET** knob

In [ ]:
import os, getpass, boto3, random, torch, numpy as np

def _secret(k):
    try:
        from google.colab import userdata
        v = userdata.get(k)
        if v: return v
    except Exception:
        pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")

os.environ["AWS_ACCESS_KEY_ID"]     = _secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
HF_TOKEN = _secret("HF_TOKEN"); os.environ["HF_TOKEN"] = HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)

BUCKET = "codec-audio-data"
s3 = boto3.client("s3", region_name=os.environ["AWS_DEFAULT_REGION"])
s3.head_bucket(Bucket=BUCKET); print("S3 connected:", BUCKET)

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEED      = 4242
SR        = 24000
LANG_CODE = "yo"                      # CONFIRM-AT-RUNTIME #4 (cell 5) re-asserts this
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert DEVICE == "cuda", "training needs the A100 — this is NOT a T4/L4 notebook"

# ====================== THE KNOBS (data-efficiency sweep) ======================
# None -> ALL clean data = the FULL run (already done: 5000 steps / 19 epochs, tone_i2 0.633 ±0.019,
#         cloning INTACT) -> the curve's RIGHT anchor; no need to re-run.
# float (15.0 / 5.0 / 1.0) -> cap TRAIN audio to ~N hours and FIX-EPOCHS-train it for the Paper-1 curve.
HOURS_BUDGET = None
# FIX-EPOCHS protocol: each budget trains to ~EPOCHS_TARGET epochs (where the full run SATURATED by
# ~epoch 9 — train loss flat after step ~2600), at the SAME batch_tokens=16384, so DATA QUANTITY is the
# only variable (fixed-steps would make 1h run hundreds of epochs = pure memorisation). Run 15.0, then
# 5.0, then 1.0. Curve = {0h zero-shot 0.598 | 1h | 5h | 15h | 28.5h full 0.633}.
EPOCHS_TARGET  = 10
CLIPS_PER_STEP = 41.3   # calibrated from the full run: 10731 clips / ~260 steps-per-epoch @ batch_tokens=16384
STEPS_OVERRIDE = None   # hard override (e.g. compute-matched runs); leave None to use EPOCHS_TARGET
INCLUDE_NAIJAVOICES = False  # clean only (bible+slr86). True needs raw wavs (see caveat).
DEV_HOURS = 0.3
# ==============================================================================
RUN_TAG = "full" if HOURS_BUDGET is None else f"{HOURS_BUDGET:g}h"
print(f"RUN_TAG={RUN_TAG} | HOURS_BUDGET={HOURS_BUDGET} | STEPS_OVERRIDE={STEPS_OVERRIDE} | NAIJA={INCLUDE_NAIJAVOICES}")

# ---- S3 layout (verbatim from project artifacts) ----
BIBLE_MANIFEST   = "tts_data/yoruba_gold/s1_train.v2.jsonl"          # clean: source=='bible' + slr86
HOLDOUTS_KEY     = "tts_data/yoruba_gold/holdouts.v1.json"           # 200 frozen eval texts (DROP from train)
S3_TONE_PREFIX   = "tts_data/yoruba/tone_v2"
F0CAL_KEY        = "tts_data/yoruba/tone_v2/f0_abs_calibration.v1.json"
CKPT_PREFIX      = f"tts_checkpoints/omnivoice_yoruba/{RUN_TAG}"     # finetuned weights land here
ABLATION_KEY     = "tts_data/yoruba/omnivoice_ablation/data_efficiency.v1.json"  # the Paper-1 table
PROBE_OUT_PREFIX = f"tts_data/yoruba/omnivoice_ft_probe/{RUN_TAG}"   # eval wavs per run

WORK = ("/content/ov_ft" if os.path.isdir("/content") else "/tmp/ov_ft")
LOCAL_WAVS = os.path.join(WORK, "wavs"); os.makedirs(LOCAL_WAVS, exist_ok=True)
TOKEN_DIR  = os.path.join(WORK, "tokens", RUN_TAG)
OUTPUT_DIR = os.path.join(WORK, "exp", f"omnivoice_yoruba_{RUN_TAG}")
os.makedirs(TOKEN_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True)
print("WORK", WORK, "| CKPT_PREFIX", CKPT_PREFIX)

## 3b. Pre-fetch HF weights — disable Xet, single-threaded (prevents the Stage-0/1 download hang)

The exact nb01 §4 failure (`hf-xet` stalling at `Fetching N files: 0%` on a fresh runtime) would
otherwise strike **inside the training subprocesses**: Stage 0 downloads the Higgs codec
(`eustlb/higgs-audio-v2-tokenizer`), Stage 1 downloads `init_from_checkpoint=k2-fsa/OmniVoice` — a
hang there wastes A100 time. We warm the HF cache here (single-threaded, Xet disabled, retried) so
both subprocesses **and** the §9 eval load get cache hits. `hf-xet` is also uninstalled in §1; the env
vars set here are belt-and-suspenders that the subprocesses inherit via `dict(os.environ, …)`.

In [ ]:
# Warm the HF cache for the two repos the training subprocesses pull, so they hit cache instead of
# the fresh-runtime hf-xet stall (the nb01 §4 hang). hf-xet is uninstalled in §1; these env vars are
# belt-and-suspenders that Stage-0/Stage-1 SUBPROCESSES inherit (they read os.environ at their own
# import). Pre-fetch is single-threaded + retried so a flaky pull errors and retries, never hangs.
import os, shutil, glob
os.environ["HF_HUB_DISABLE_XET"]        = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]   = "60"
os.environ["HF_HUB_ETAG_TIMEOUT"]       = "60"
try:
    import huggingface_hub.constants as _hc; _hc.HF_HUB_DISABLE_XET = True
except Exception:
    pass

from huggingface_hub import snapshot_download
HUB = os.path.join(os.path.expanduser(os.environ.get("HF_HOME", "~/.cache/huggingface")), "hub")
_locks = os.path.join(HUB, ".locks")
if os.path.isdir(_locks):
    shutil.rmtree(_locks, ignore_errors=True); print("cleared stale .locks")

def _prefetch(repo):
    md = os.path.join(HUB, "models--" + repo.replace("/", "--"))
    for f in glob.glob(os.path.join(md, "blobs", "*.incomplete")):
        try: os.remove(f)
        except OSError: pass
    last = None
    for k in range(1, 4):
        try:
            p = snapshot_download(repo, max_workers=1, etag_timeout=60)  # single-threaded, resumable
            print(f"cached {repo} -> {p}"); return p
        except Exception as e:
            last = e; print(f"{repo} attempt {k}/3 failed: {type(e).__name__}: {e}")
    raise RuntimeError(f"could not pre-fetch {repo} after 3 tries — just re-run this cell") from last

OV_LOCAL    = _prefetch("k2-fsa/OmniVoice")                  # Stage 1: init_from_checkpoint
HIGGS_LOCAL = _prefetch("eustlb/higgs-audio-v2-tokenizer")   # Stage 0: --tokenizer_path + §9 codec
assert os.path.isdir(os.path.join(OV_LOCAL, "audio_tokenizer")), \
    "k2-fsa/OmniVoice snapshot is missing audio_tokenizer/ — codec load would re-hit the network"
print("HF pre-fetch done — Stage 0/1 + §9 eval will hit cache (no Xet stall).")

## 4. Build the OmniVoice finetune JSONL (bible wavs from S3 → local), apply HOURS_BUDGET + holdout drop

In [ ]:
import io, json, random, unicodedata
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def norm_key(s):  # MUST match nb29's holdout normalization
    return " ".join(unicodedata.normalize("NFC", s).lower().split())

# (a) frozen eval texts -> never train on them
HELD = set()
try:
    hold = json.loads(s3.get_object(Bucket=BUCKET, Key=HOLDOUTS_KEY)["Body"].read())
    def collect(x):
        if isinstance(x, str): HELD.add(norm_key(x))
        elif isinstance(x, dict): [collect(v) for v in x.values()]
        elif isinstance(x, list): [collect(v) for v in x]
    for k in ("eval_texts", "minimal_pairs"):
        if k in hold: collect(hold[k])
    print("holdout eval-text keys:", len(HELD))
except Exception as e:
    print("WARNING: holdouts not loaded (%s) -> NO holdout drop (NOT safe for the ablation eval)" % e)

# (b) stream the clean manifest; collect candidates
cands = []
body = s3.get_object(Bucket=BUCKET, Key=BIBLE_MANIFEST)["Body"]
for raw in io.TextIOWrapper(body, encoding="utf-8"):
    r = json.loads(raw)
    if norm_key(r["text"]) in HELD:           # holdout drop
        continue
    cands.append(dict(id=r["id"], text=r["text"], dur=float(r["duration_sec"]),
                      wav_key=r["audio_s3_key"]))   # bible -> wav_biblefull/, slr86 -> wav/
print(f"clean candidate clips (post-holdout): {len(cands)} | {sum(c['dur'] for c in cands)/3600:.2f} h")

# (c) optional NaijaVoices — needs a raw per-utterance wav (see cell-8 caveat); off by default
if INCLUDE_NAIJAVOICES:
    raise NotImplementedError(
        "NaijaVoices rows carry pre-encoded audio_codes + a ref wav, NOT a per-utterance training wav. "
        "Resolve a raw audio_path per row before enabling. Default off.")

# (d) deterministic shuffle, carve dev, then cap to HOURS_BUDGET (print what is dropped — no silent cap)
rng = random.Random(SEED); rng.shuffle(cands)
def take_hours(rows, budget_h):
    if budget_h is None: return rows, sum(c["dur"] for c in rows)
    acc, out = 0.0, []
    for c in rows:
        if acc >= budget_h * 3600: break
        out.append(c); acc += c["dur"]
    return out, acc

dev_rows, dev_s = take_hours(cands, DEV_HOURS)
rest = cands[len(dev_rows):]
train_rows, train_s = take_hours(rest, HOURS_BUDGET)
dropped_n = len(rest) - len(train_rows)
dropped_h = (sum(c["dur"] for c in rest) - train_s) / 3600
print(f"DEV  : {len(dev_rows)} clips / {dev_s/3600:.3f} h")
print(f"TRAIN: {len(train_rows)} clips / {train_s/3600:.3f} h (budget={HOURS_BUDGET})")
print(f"DROPPED by budget: {dropped_n} clips / {dropped_h:.2f} h "
      f"({'none — full run' if HOURS_BUDGET is None else 'expected for ablation'})")
assert len(train_rows) > 0, "no train rows after budget — check HOURS_BUDGET"

# (e) download wavs concurrently; set local audio_path
def _dl(c):
    local = os.path.join(LOCAL_WAVS, c["id"] + ".wav")
    if not os.path.exists(local):
        s3.download_file(BUCKET, c["wav_key"], local)
    c["audio_path"] = local
    return c
all_rows = train_rows + dev_rows
with ThreadPoolExecutor(max_workers=32) as ex:
    list(tqdm(ex.map(_dl, all_rows), total=len(all_rows), desc="download wavs"))

# (f) write the two OmniVoice finetune JSONLs — EXACT keys per verdict
def write_manifest(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for c in rows:
            f.write(json.dumps({
                "id":          c["id"],
                "audio_path":  c["audio_path"],   # LOCAL fs path (verdict-confirmed key)
                "text":        c["text"],
                "language_id": LANG_CODE,          # "yo"
                "duration":    round(c["dur"], 3), # optional passthrough (Stage 0 ignores unknown keys)
            }, ensure_ascii=False) + "\n")
TRAIN_JSONL = os.path.join(WORK, f"train_{RUN_TAG}.jsonl")
DEV_JSONL   = os.path.join(WORK, f"dev_{RUN_TAG}.jsonl")
write_manifest(train_rows, TRAIN_JSONL); write_manifest(dev_rows, DEV_JSONL)
print("wrote", TRAIN_JSONL, "and", DEV_JSONL)
print("sample line:", open(TRAIN_JSONL).readline().strip()[:200])

## 5. Stage 0 — tokenize wavs with the Higgs codec → WebDataset shards + data.lst

In [ ]:
import subprocess, os, sys

GPU_IDS = "0"      # single A100 (run_finetune.sh default is "0,1"; we override to 1 GPU)
TOKENIZER_PATH = "eustlb/higgs-audio-v2-tokenizer"   # Higgs/Boson codec (verdict default; license caveat)

def stage0(split, split_jsonl):
    out_tar   = os.path.join(TOKEN_DIR, split, "audios", "shard-%06d.tar")
    out_jsonl = os.path.join(TOKEN_DIR, split, "txts",   "shard-%06d.jsonl")
    os.makedirs(os.path.dirname(out_tar), exist_ok=True)
    os.makedirs(os.path.dirname(out_jsonl), exist_ok=True)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU_IDS, PYTHONPATH=OV_DIR)
    cmd = [sys.executable, "-m", "omnivoice.scripts.extract_audio_tokens",
           "--input_jsonl", split_jsonl,
           "--tar_output_pattern", out_tar,
           "--jsonl_output_pattern", out_jsonl,
           "--tokenizer_path", TOKENIZER_PATH,
           "--nj_per_gpu", "3",          # verbatim from run_finetune.sh
           "--shuffle", "True"]
    print(">>", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=OV_DIR, env=env)
    lst = os.path.join(TOKEN_DIR, split, "data.lst")
    assert os.path.exists(lst), f"Stage 0 did not write {lst} — check shard output"
    print(f"[{split}] data.lst lines (shards): {sum(1 for _ in open(lst))}  ->", lst)
    return lst

TRAIN_LST = stage0("train", TRAIN_JSONL)
DEV_LST   = stage0("dev",   DEV_JSONL)

## 6. Write the finetune configs (derived from the introspected stock config) — sdpa, 1 GPU, single-language

In [ ]:
import json, os

# ---- data_config : point at OUR token dirs (shape from recon) ----
DATA_CFG = {"train": [{"manifest_path": [TRAIN_LST]}],
            "dev":   [{"manifest_path": [DEV_LST]}]}
DATA_CFG_PATH = os.path.join(WORK, f"data_config_{RUN_TAG}.json")
json.dump(DATA_CFG, open(DATA_CFG_PATH, "w"), indent=2)

# ---- train config : START from the introspected stock config (cell 5), override only what's needed ----
cfg = dict(stock)
approx_clips = len(train_rows)
if STEPS_OVERRIDE:
    steps = int(STEPS_OVERRIDE)
elif HOURS_BUDGET is None:
    steps = 5000                                # FULL run: already done at 5000 steps (saturated by ~ep 9)
else:
    # FIX-EPOCHS: same ~EPOCHS_TARGET epochs for every budget; steps/epoch = clips / CLIPS_PER_STEP
    steps = max(50, round(EPOCHS_TARGET * approx_clips / CLIPS_PER_STEP))
save_eval = max(50, steps // 8)                 # keep all checkpoints (keep_last_n=-1) for tone_i2 selection

cfg.update({
    "init_from_checkpoint": "k2-fsa/OmniVoice",   # HF Hub id -> snapshot_download (verified)
    "resume_from_checkpoint": None,
    "attn_implementation": "sdpa",                # <-- REQUIRED override (no flex/flash)
    "learning_rate": 1e-5,                        # OmniVoice finetune default (collapse-safe by design)
    "steps": steps,
    "warmup_type": "ratio", "warmup_ratio": 0.01, "warmup_steps": 0,
    "batch_tokens": 16384,                         # locked for the sweep (matches the full run)
    "gradient_accumulation_steps": 1,
    "mixed_precision": "bf16",
    "allow_tf32": True,
    "num_workers": 2,
    "logging_steps": max(10, save_eval // 5),
    "eval_steps": save_eval,
    "save_steps": save_eval,
    "keep_last_n_checkpoints": -1,                # keep all (ablation wants final + curve)
    "seed": SEED,
    "language_ratio": 1.0,                        # single-language: always tag 'yo' (stock default 0.8)
})
# sdpa-variant extras (harmless: TrainingConfig.from_json drops unknown keys) — only if stock lacks them
for k, v in {"max_sample_tokens": 2000, "min_sample_tokens": 50, "max_batch_size": 64}.items():
    cfg.setdefault(k, v)

TRAIN_CFG_PATH = os.path.join(WORK, f"train_config_{RUN_TAG}.json")
json.dump(cfg, open(TRAIN_CFG_PATH, "w"), indent=2)
print("=== data_config ===\n", json.dumps(DATA_CFG, indent=2))
print("\n=== train_config (FULL, written to disk) ===\n", json.dumps(cfg, indent=2))
print(f"\nsteps={steps} save/eval={save_eval} | train clips≈{approx_clips} | "
      f"~{steps*CLIPS_PER_STEP/max(approx_clips,1):.1f} epochs"
      f"{'' if (STEPS_OVERRIDE or HOURS_BUDGET is None) else f' (target {EPOCHS_TARGET})'}")

## 7. Stage 1 — finetune (accelerate launch, 1×A100, sdpa). Streams logs live.

In [ ]:
import subprocess, os, sys, glob

NUM_GPUS = 1
env = dict(os.environ, PYTHONPATH=OV_DIR)
cmd = ["accelerate", "launch",
       "--gpu_ids", GPU_IDS,            # "0"
       "--num_processes", str(NUM_GPUS),
       "-m", "omnivoice.cli.train",
       "--train_config", TRAIN_CFG_PATH,
       "--data_config",  DATA_CFG_PATH,
       "--output_dir",   OUTPUT_DIR]
print(">>", " ".join(cmd), "\n")
proc = subprocess.Popen(cmd, cwd=OV_DIR, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
assert rc == 0, f"training exited {rc} — inspect logs above"

ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
               key=lambda p: int(p.rsplit("-", 1)[-1]))
print("\ncheckpoints:", [os.path.basename(c) for c in ckpts])
assert ckpts, "no checkpoint-* dir written — check save_steps vs steps"
FINAL_CKPT = ckpts[-1]
print("FINAL_CKPT:", FINAL_CKPT)

## 8. Upload the finetuned checkpoint → `s3://…/tts_checkpoints/omnivoice_yoruba/<hours>/`

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def _upload_dir(local_dir, s3_prefix):
    files = []
    for root, _, fns in os.walk(local_dir):
        for fn in fns:
            lp = os.path.join(root, fn)
            files.append((lp, f"{s3_prefix}/{os.path.relpath(lp, local_dir)}"))
    def _put(t): s3.upload_file(t[0], BUCKET, t[1]); return t[1]
    with ThreadPoolExecutor(max_workers=16) as ex:
        return list(tqdm(ex.map(_put, files), total=len(files), desc="upload ckpt"))

ckpt_keys = _upload_dir(FINAL_CKPT, CKPT_PREFIX)
s3.upload_file(TRAIN_CFG_PATH, BUCKET, f"{CKPT_PREFIX}/train_config_used.json")  # reproducibility
print(f"uploaded {len(ckpt_keys)} files -> s3://{BUCKET}/{CKPT_PREFIX}/")

## 9. EVAL the finetuned checkpoint on the SAME nb01 gate (cer / tone_i2 / tone_i2_cov / ssim / len_ratio)

In [ ]:
import torch, json, io, random, soundfile as sf
from omnivoice import OmniVoice
from transformers import AutoModel, AutoFeatureExtractor

# (a) the FINETUNED model (local dir; load via OmniVoice's own loader — never guess weight filenames)
ovft = OmniVoice.from_pretrained(FINAL_CKPT,
                                 device_map=("cuda:0" if DEVICE == "cuda" else "cpu"),
                                 dtype=torch.float16)
OV_SR = int(getattr(ovft, "sampling_rate", 0) or SR)
assert OV_SR == 24000, f"CONFIRM: expected 24000, got {OV_SR}"

# (b) gate stack — identical to nb01
rm = RewardModels(device=DEVICE); assert rm.ecapa is not None
enc = AutoModel.from_pretrained("ajesujoba/AfriHuBERT").to(DEVICE).eval()
fe  = AutoFeatureExtractor.from_pretrained("ajesujoba/AfriHuBERT")
objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{S3_TONE_PREFIX}/tone_probe_L").get("Contents")
probe_key = sorted(o["Key"] for o in objs)[-1]
s3.download_file(BUCKET, probe_key, f"{WORK}/tone_probe.pt")
PROBE, PMETA = tp.load_probe(f"{WORK}/tone_probe.pt", DEVICE); LAYER = PMETA.get("layer", 9)
s3.download_file(BUCKET, F0CAL_KEY, f"{WORK}/f0cal.json")
F0CAL = json.load(open(f"{WORK}/f0cal.json"))
I2_TH, I2_TL = F0CAL["theta_h"], F0CAL["theta_l"]
I2_MODE, I2_LATE = F0CAL.get("mode", "blind"), F0CAL.get("late_frac", 0.5)

# (c) probe set — minimal pairs + N long held-out + ONE bible ref (nb01 cell 13 verbatim)
MP = json.load(open(os.path.join(CODE_DIR, "minimal_pairs_draft.json")))
CARRIER = MP["carriers"][0]["template"]
probe_lines = []
for s_ in MP["sets"]:
    for j, it in enumerate(s_["items"]):
        probe_lines.append(dict(id=f"mp_{s_['base']}_{j}", kind="minpair", base=s_["base"],
                                word=it["text"], tones=it.get("tones"),
                                text=CARRIER.replace("___", it["text"])))
HOLD = json.loads(s3.get_object(Bucket=BUCKET, Key=HOLDOUTS_KEY)["Body"].read())
EVAL_ALL = list(HOLD["eval_texts"]); rng = random.Random(SEED); N_LONG = 8
for k, e in enumerate(rng.sample(EVAL_ALL, N_LONG)):
    probe_lines.append(dict(id=f"long_{k}_{e['base']}", kind="long", base=e["base"],
                            word=None, tones=None, text=e["text"]))
ref_row = None
for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET, Key=BIBLE_MANIFEST)["Body"], encoding="utf-8"):
    r = json.loads(raw)
    if r.get("source") == "bible" and 3.0 <= float(r.get("duration_sec", 0)) <= 10.0:
        ref_row = r; break
REF_WAV_PATH = f"{WORK}/ref.wav"; s3.download_file(BUCKET, ref_row["audio_s3_key"], REF_WAV_PATH)
REF_TEXT = ref_row["text"]; ref_wav, ref_sr = sf.read(REF_WAV_PATH, dtype="float32")
if ref_wav.ndim > 1: ref_wav = ref_wav.mean(-1)
print("eval ready:", len(probe_lines), "probe lines; ref", ref_row["audio_s3_key"])

In [ ]:
import soundfile as sf, numpy as np
import IPython.display as ipd
from tqdm.auto import tqdm
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

syn = []
for p in tqdm(probe_lines, desc="OmniVoice-FT synth"):
    audio = ovft.generate(text=p["text"], language=LANG_CODE, ref_audio=REF_WAV_PATH, ref_text=REF_TEXT)
    w = np.asarray(audio[0], dtype="float32")
    local = f"{WORK}/ft_{p['id']}.wav"; sf.write(local, w, OV_SR)
    syn.append(dict(**p, wav=w, fs=OV_SR, local=local))

def _bal_tone_acc(pairs):
    if not pairs: return float("nan")
    tot, cor = defaultdict(int), defaultdict(int)
    for pp, tt in pairs: tot[tt] += 1; cor[tt] += int(pp == tt)
    recs = [cor[c]/tot[c] for c in tot if tot[c] > 0]
    return float(np.mean(recs)) if recs else float("nan")

def score_one(rec):
    wav, fs, text = rec["wav"], rec["fs"], rec["text"]
    logits, n16 = rm.asr_logits(wav, fs)
    cer = RewardModels.cer(text, rm.decode_logits(logits))
    i1 = tp.probe_score(wav, fs, text, PROBE, enc, fe, asr=rm.asr, proc=rm.asr_proc,
                        device=DEVICE, emissions=logits, n16=n16, layer=LAYER)
    i2 = f0a.f0_abs_score(wav, fs, text, asr=rm.asr, proc=rm.asr_proc, device=DEVICE,
                          emissions=logits, n16=n16, theta_h=I2_TH, theta_l=I2_TL,
                          mode=I2_MODE, mid_ref=None, late_frac=I2_LATE)
    pairs = [(pp, tt) for pp, tt in zip(i2["pred"], i2["target"]) if pp is not None]
    return dict(id=rec["id"], kind=rec["kind"], text=text, cer=cer,
                i1_acc=i1["accuracy"], i1_cov=i1["coverage"],
                tone_i2=_bal_tone_acc(pairs), tone_i2_cov=i2["coverage"],
                ssim=rm.ssim(wav, fs, ref_wav, ref_sr),
                len_ratio=(len(wav)/fs)/max(0.157*len(text), 1e-6),
                wav=rec["wav"], fs=rec["fs"], local=rec["local"],   # carry audio so we can upload + listen inline
                i2_pairs=pairs)

scored = [score_one(r) for r in tqdm(syn, desc="score")]
allpairs = [pr for r in scored for pr in r["i2_pairs"]]
val = lambda k: [r[k] for r in scored if r[k] == r[k]]
AGG = dict(n=len(scored), cer=float(np.mean(val("cer"))),
           tone_i2=_bal_tone_acc(allpairs),
           tone_i2_cov=float(np.mean([r["tone_i2_cov"] for r in scored])),
           i1_acc=float(np.mean(val("i1_acc"))),
           ssim=float(np.mean(val("ssim"))),
           len_ratio=float(np.mean([r["len_ratio"] for r in scored])))
print(RUN_TAG, "AGG:", json.dumps(AGG, indent=2))

# (a) upload to S3 (durable; native ear can also stream them from there)
with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(lambda r: s3.upload_file(r["local"], BUCKET,
                f"{PROBE_OUT_PREFIX}/wav/{r['id']}.wav"), scored))
print("eval wavs ->", f"s3://{BUCKET}/{PROBE_OUT_PREFIX}/wav/")

# (b) ALSO play inline for a fast in-notebook listen (from the in-kernel arrays; no S3 round-trip)
print("\n=== inline preview (native ear is decisive) ===")
preview = [r for r in scored if r["kind"] == "minpair"][:5] + [r for r in scored if r["kind"] == "long"][:3]
for r in preview:
    print(f'{r["id"]:26} | cer {r["cer"]:.2f}  tone_i2 {r["tone_i2"]:.2f} | {r["text"][:52]}')
    ipd.display(ipd.Audio(r["wav"], rate=r["fs"]))


## 10. Append this run to the data-efficiency table (Paper-1 ablation artifact)

In [ ]:
import json, datetime

try:
    TABLE = json.loads(s3.get_object(Bucket=BUCKET, Key=ABLATION_KEY)["Body"].read())
except s3.exceptions.NoSuchKey:
    TABLE = {"meta": {"axis": "data-efficiency (clean Yoruba hours -> tone_i2, cer)",
                      "model": "k2-fsa/OmniVoice finetune", "gate": "nb01 (tone_i2 F0-abs + MMS CER)",
                      "lang_code": LANG_CODE, "seed": SEED}, "runs": {}}

train_hours = round(sum(c["dur"] for c in train_rows)/3600, 3)   # ACTUAL hours used (honest x-axis)
TABLE["runs"][RUN_TAG] = {
    "hours_budget": HOURS_BUDGET, "train_hours_actual": train_hours, "train_clips": len(train_rows),
    "include_naijavoices": INCLUDE_NAIJAVOICES,
    "steps": cfg["steps"], "lr": cfg["learning_rate"], "attn": cfg["attn_implementation"],
    "ckpt_s3": f"s3://{BUCKET}/{CKPT_PREFIX}/", "eval_wavs_s3": f"s3://{BUCKET}/{PROBE_OUT_PREFIX}/wav/",
    "tone_i2": AGG["tone_i2"], "tone_i2_cov": AGG["tone_i2_cov"], "cer": AGG["cer"],
    "ssim": AGG["ssim"], "len_ratio": AGG["len_ratio"], "i1_acc": AGG["i1_acc"],
    "date": datetime.date.today().isoformat(),
}
s3.put_object(Bucket=BUCKET, Key=ABLATION_KEY,
              Body=json.dumps(TABLE, ensure_ascii=False, indent=2).encode("utf-8"))

print(f"=== data-efficiency curve (s3://{BUCKET}/{ABLATION_KEY}) ===")
hdr = f"{'run':>6} {'hrs':>7} {'clips':>6} {'tone_i2':>8} {'cer':>6} {'ssim':>6} {'lenR':>6}"
print(hdr); print("-" * len(hdr))
for tag, r in sorted(TABLE["runs"].items(), key=lambda kv: kv[1]["train_hours_actual"]):
    print(f"{tag:>6} {r['train_hours_actual']:>7.2f} {r['train_clips']:>6} "
          f"{r['tone_i2']:>8.3f} {r['cer']:>6.3f} {r['ssim']:>6.3f} {r['len_ratio']:>6.3f}")
print("\nZero-shot anchor (nb01 GO/NO-GO probe): the SAME I2 meter on the SAME held-out set -> "
      "every row is directly comparable to zero-shot and across hours.")

## 11. Reading this against Qwen3-TTS (the OTHER axis) — line them up, don't rebuild here

This notebook produces ONE axis: **data efficiency** (clean Yoruba hours → tone_i2/cer) for a
**non-autoregressive masked-diffusion** TTS (OmniVoice). The **second, orthogonal axis** —
**architecture: AR vs non-AR** — is owned by the Qwen3-TTS experiments. Do **not** rebuild it here.

**How to line the two up for the paper (no new code here):**
- **Same gate, same set.** Both stacks score on the identical I2 F0-absolute tone meter
  (`tone_f0_abs.py`) + MMS CER + ECAPA SSIM, on the **same frozen `holdouts.v1.json` eval texts** and
  the **same bible reference clip** → `tone_i2`/`cer`/`ssim` are comparable with zero re-normalization.
- **AR (Qwen3-TTS)** points come from nb26 (`26_s1_bakeoff.ipynb` → `s1_verdict.json`): collapsed
  high-LR `tone_i2 ≈ 0.398`, underfit low-LR flat.
- **non-AR (OmniVoice)** points come from THIS notebook's `data_efficiency.v1.json["runs"]`.
- **Paper table** = a 2×N grid: rows {AR Qwen3-TTS, non-AR OmniVoice}; columns = matched hours.
  Read off: (a) does non-AR avoid the AR collapse? (b) at equal hours, which carries more tone?
- **Not controlled (state honestly):** different codecs (Higgs/Boson vs Qwen3-TTS-Tokenizer),
  backbones (Qwen3-0.6B vs Qwen3-1.7B-Base), lr regimes (1e-5 vs 2e-6/2e-5). The per-architecture
  data-efficiency curve is clean; the cross-architecture comparison is **suggestive, not a controlled A/B**.

## 12. Post-finetune capability checks — did the finetune damage instruct or cloning?

Single-language finetuning on the ~single-speaker BibleTTS corpus has two failure modes the main eval
doesn't catch (it clones a *bible* ref): **(A)** catastrophic forgetting of the English-trained
**voice-design** (`instruct`) path, and **(B)** **cloning collapse** — the model ignoring `ref_audio`
and always sounding like the bible narrator, which would quietly kill flywheel speaker diversity.
Both run on the **finetuned** model `ovft`; reuse the gate stack from §9.

In [ ]:
# (A) INSTRUCT SURVIVAL — does the FINETUNED model still do voice-design, and does Yoruba tone survive?
# NOTE: OmniVoice instruct = CONTROLLED attribute tags (validated vs _INSTRUCT_VALID_EN, split on
# re.split(r"\s*[,，]\s*", ...)), NOT free-form prose. Self-verify tags before generating.
import numpy as np, re as _re
try:
    from omnivoice.utils.voice_design import _INSTRUCT_VALID_EN
    _VALID = set(_INSTRUCT_VALID_EN)
except Exception:   # fallback to the vocab OmniVoice prints in its validation error
    _VALID = {"american accent","australian accent","british accent","canadian accent","child",
              "chinese accent","elderly","female","high pitch","indian accent","japanese accent",
              "korean accent","low pitch","male","middle-aged","moderate pitch","portuguese accent",
              "russian accent","teenager","very high pitch","very low pitch","whisper","young adult"}
INSTRUCTS = ["female, young adult, high pitch", "male, elderly, low pitch"]   # valid tags, no accent
for _d in INSTRUCTS:
    _bad = [t for t in _re.split(r"\s*[,，]\s*", _d.lower()) if t and t not in _VALID]
    assert not _bad, f"invalid instruct tag(s) {_bad} in '{_d}'. Valid English tags: {sorted(_VALID)}"

inst_lines = [p for p in probe_lines if p["kind"] == "minpair"][:8]
syn_i, inst_fail = [], []
for di, desc in enumerate(INSTRUCTS):
    for p in inst_lines:
        try:
            a = ovft.generate(text=p["text"], language=LANG_CODE, instruct=desc)   # no ref_audio
            syn_i.append(dict(text=p["text"], wav=np.asarray(a[0], dtype="float32"), fs=OV_SR))
        except Exception as ex:
            inst_fail.append((p["id"], di, f"{type(ex).__name__}: {ex}"))
print(f"instruct(ft) clips: {len(syn_i)}; failed: {len(inst_fail)}")
for fid, di, m in inst_fail[:3]: print("  FAIL", fid, di, "->", m)

def _cer_pairs(rec):
    logits, n16 = rm.asr_logits(rec["wav"], rec["fs"])
    cer = RewardModels.cer(rec["text"], rm.decode_logits(logits))
    i2 = f0a.f0_abs_score(rec["wav"], rec["fs"], rec["text"], asr=rm.asr, proc=rm.asr_proc, device=DEVICE,
                          emissions=logits, n16=n16, theta_h=I2_TH, theta_l=I2_TL,
                          mode=I2_MODE, mid_ref=None, late_frac=I2_LATE)
    return cer, [(pp, tt) for pp, tt in zip(i2["pred"], i2["target"]) if pp is not None]

INSTRUCT_FT = None
if syn_i:
    cers, allpairs = [], []
    for r in syn_i:
        c, pr = _cer_pairs(r); cers.append(c); allpairs += pr
    INSTRUCT_FT = dict(n=len(syn_i), cer=float(np.mean(cers)), tone_i2=_bal_tone_acc(allpairs))
    print(f"\ninstruct(ft): cer {INSTRUCT_FT['cer']:.3f} | tone_i2 {INSTRUCT_FT['tone_i2']:.3f}   "
          f"vs CLONE(ft): cer {AGG['cer']:.3f} | tone_i2 {AGG['tone_i2']:.3f}")
    print("  -> instruct tone_i2 ~= clone tone_i2  => voice-design SURVIVED the finetune.")
    print("  -> instruct tone_i2 << clone           => finetune hurt voice-design (use cloning for the flywheel).")
else:
    print("instruct mode unavailable on this checkpoint (all calls failed) -> rely on cloning for speaker control.")

In [ ]:
# (B) CLONING DIVERSITY — after finetuning on ~single-speaker BibleTTS, does the model still ADOPT a
#     HELD-OUT non-bible voice, or has it collapsed toward the bible narrator?  (mirrors nb26's voice gate)
import json, io, numpy as np, soundfile as sf

NV = [json.loads(l) for l in io.TextIOWrapper(
        s3.get_object(Bucket=BUCKET, Key="tts_data/yoruba/manifest.v3.jsonl")["Body"], encoding="utf-8")]
heldset = set(HOLD.get("naijavoices_holdout_speakers", []))
by_spk = {}
for r in NV:
    if r.get("speaker_id") in heldset and r.get("text"):
        by_spk.setdefault(r["speaker_id"], []).append(r)

VOICES = []
for spk in list(by_spk)[:3]:                                   # 3 held-out non-bible voices
    r = sorted(by_spk[spk], key=lambda x: x["audio_s3_key"])[0]  # first clip (nb26 convention)
    p = f"{WORK}/nb_ref_{spk}.wav"; s3.download_file(BUCKET, r["audio_s3_key"], p)
    rw, rsr = sf.read(p, dtype="float32"); rw = rw.mean(-1) if rw.ndim > 1 else rw
    VOICES.append(dict(spk=spk, ref_path=p, ref_text=r["text"], ref_wav=rw, ref_sr=rsr))

NONBIBLE_FT = None
if not VOICES:
    print("WARNING: no held-out NaijaVoices speaker (with text) resolved from manifest.v3.jsonl -> "
          "skipping non-bible clone check. Verify holdouts['naijavoices_holdout_speakers'] + manifest schema.")
else:
    test_lines = ([p for p in probe_lines if p["kind"] == "long"][:3] or probe_lines[:3])
    rows = []
    for v in VOICES:
        s_voice, s_bible = [], []
        for p in test_lines:
            a = ovft.generate(text=p["text"], language=LANG_CODE,
                              ref_audio=v["ref_path"], ref_text=v["ref_text"])
            w = np.asarray(a[0], dtype="float32")
            s_voice.append(rm.ssim(w, OV_SR, v["ref_wav"], v["ref_sr"]))   # adopted the held-out voice?
            s_bible.append(rm.ssim(w, OV_SR, ref_wav, ref_sr))             # or collapsed to bible?
        rows.append(dict(spk=v["spk"], to_voice=float(np.mean(s_voice)), to_bible=float(np.mean(s_bible))))
    print(f"{'held-out spk':18} {'SSIM->voice':>11} {'SSIM->bible':>11}")
    for r in rows:
        flag = "  ok" if r["to_voice"] > r["to_bible"] else "  <-- leaning to bible"
        print(f"{r['spk'][:18]:18} {r['to_voice']:>11.3f} {r['to_bible']:>11.3f}{flag}")
    mv = float(np.mean([r["to_voice"] for r in rows])); mb = float(np.mean([r["to_bible"] for r in rows]))
    NONBIBLE_FT = dict(n_voices=len(rows), ssim_to_voice=mv, ssim_to_bible=mb,
                       intact=bool(mv > mb + 0.05))
    print(f"\nmean SSIM ->held-out-voice {mv:.3f}  vs  ->bible {mb:.3f}  ->  "
          f"{'cloning diversity INTACT' if NONBIBLE_FT['intact'] else 'WARNING: voice collapsing toward bible'}")

# persist both capability checks alongside the run's eval
import datetime
CAP = dict(run_tag=RUN_TAG, date=datetime.date.today().isoformat(),
           instruct_ft=INSTRUCT_FT, nonbible_clone_ft=NONBIBLE_FT,
           clone_ft=dict(tone_i2=AGG["tone_i2"], cer=AGG["cer"], ssim=AGG["ssim"]))
s3.put_object(Bucket=BUCKET, Key=f"{PROBE_OUT_PREFIX}/capability.json",
              Body=json.dumps(CAP, ensure_ascii=False, indent=2).encode("utf-8"))
print(f"\ncapability checks -> s3://{BUCKET}/{PROBE_OUT_PREFIX}/capability.json")

### How to read these (per ablation point)

- **Instruct survival:** if `instruct(ft) tone_i2 ≈ clone(ft) tone_i2`, voice-design survived the
  finetune → you can mint flywheel speakers by *description*. If it cratered, finetuning forgot the
  English-trained voice-design path → mint speakers by *cloning* instead (no loss — cloning is the
  primary mechanism and is reinforced by finetuning).
- **Cloning diversity:** `SSIM→held-out-voice` should clearly beat `SSIM→bible`. If they converge (or
  bible wins), the model is **collapsing toward the single bible narrator** — the real danger for a
  flywheel that needs many speakers. Mitigation: use a *lighter* finetune (fewer hours/steps — the
  ablation's small-budget points already do this) and/or add multi-speaker data later. Watch this
  number across the data-efficiency sweep: more hours of single-speaker bible = more collapse risk.